<a href="https://colab.research.google.com/github/OishiNikku/GPT-TS/blob/main/notebooks/pka_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calculate microstate pKa values

Code and more documentation at:
https://github.com/mayrf/pkasolver

In [1]:
# @title Initializing Condacolab
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [2]:
# @title Check that everything is fine

import condacolab
condacolab.check()

✨🍰✨ Everything looks OK!


In [3]:
# @title Installing dependencies and pkasolver package (this might take up to 5 minutes)

print('📦 Installing dependencies ...')
!mamba install -c conda-forge rdkit > /dev/null
print('🔥 Installing PyTorch and PyG ...')
!pip install torch==1.13.1+cpu -f https://download.pytorch.org/whl/cpu/torch_stable.html > /dev/null
!pip install torch-scatter -f https://data.pyg.org/whl/torch-1.13.1+cpu.html > /dev/null
!pip install torch-sparse -f https://data.pyg.org/whl/torch-1.13.1+cpu.html > /dev/null
!pip install torch-spline-conv torch-geometric==2.0.1 -f https://data.pyg.org/whl/torch-1.13.1+cpu.html > /dev/null
!pip install cairosvg svgutils molvs > /dev/null
print('✔️ Installing pkasolver package ...')
!pip install -q git+https://github.com/mayrf/pkasolver.git > /dev/null
print("🎉 Done!")

📦 Installing dependencies ...
🔥 Installing PyTorch and PyG ...
✔️ Installing pkasolver package ...
🎉 Done!


In [26]:
# @title Predict pKa values
from pkasolver.query import QueryModel
from pkasolver.ml_architecture import GINPairV1
import pickle
import pkasolver
import torch
from os import path
from rdkit import Chem
from pkasolver.query import calculate_microstate_pka_values, draw_pka_reactions
from IPython.display import display
import pandas as pd

# load trained model
base_path = path.dirname(pkasolver.__file__)
# get input

buffers_smiles = pd.read_csv("/content/buffers_smiles.csv", encoding='latin-1')

#buffers_smiles.at[1, 'Pka']=10
#print(buffers_smiles.head())

for i in range(0, len(buffers_smiles) - 1):
  smiles = buffers_smiles.at[i, "SMILES"]
  if buffers_smiles.at[i, "Source"] == "PubChem":
    print("Passed: " + smiles + "\n")
    continue
  pka = ""
  print("Now calculating:")
  print(smiles)
  # convert from Smiles to rdkit mol
  mol = Chem.MolFromSmiles(smiles)
  ################################################
  ################################################
  # calculate microstate pka values
  protonation_states = calculate_microstate_pka_values(mol, only_dimorphite=False)
  ################################################
  try:
    for i in range(len(protonation_states)):
      pka = pka + "pKa" + str(i + 1) + " " + str(protonation_states[i].pka) + " ± " + str(protonation_states[i].pka_stddev) + "; "
  except TypeError:
    pka = pka + "pKa" + " " + str(protonation_states.pka) + " ± " + str(protonation_states.pka_stddev)


	# # Look up pKa using pka_lookup_pubchem():
	# print(f'pKa from Pubchem using smiles:\n{pka_lookup_pubchem(smiles_string)}')
  print('Calculated pKa: ' + str(pka))
  buffers_smiles.at[i, 'Pka'] = pka
  buffers_smiles.at[i, 'Source'] = "Calculated with pKaSolver"
buffers_smiles.to_csv('buffers_smiles.csv')




Now calculating:
CC(C)C[C@@H](O)C(O)=O


[query.py:297 - calculate_microstate_pka_values()] Using dimorphite-dl to identify protonation sites.


Proposed mol at pH 7.4: CC(C)C[C@@H](O)C(=O)[O-]
Calculated pKa: pKa1 4.0909154510498045 ± 0.23263259717857646; pKa2 10.247186546325684 ± 0.8306138285761503; 
Now calculating:
O[C@@H](CCC(O)=O)C(O)=O


[query.py:297 - calculate_microstate_pka_values()] Using dimorphite-dl to identify protonation sites.


Proposed mol at pH 7.4: O=C([O-])CC[C@H](O)C(=O)[O-]
Calculated pKa: pKa1 3.862264699935913 ± 0.26956062763660094; pKa2 4.525524082183838 ± 0.3678276883451202; pKa3 10.031268882751466 ± 1.1792258999684857; 
Now calculating:
OC(C(O)C(O)=O)C(O)=O


...

Proposed mol at pH 7.4: O=C([O-])CCC(O)CC(=O)[O-]
Calculated pKa: pKa1 4.158293561935425 ± 0.19994691257717945; pKa2 4.849744815826416 ± 0.23375798239018428; pKa3 9.922862167358398 ± 1.2702680135910884; 
Passed: OC(=O)C1=CC=CN=C1C(O)=O

Now calculating:
OC(=O)C(=O)CC1=CC=C(O)C=C1


[query.py:297 - calculate_microstate_pka_values()] Using dimorphite-dl to identify protonation sites.


Proposed mol at pH 7.4: O=C([O-])C(=O)Cc1ccc(O)cc1
Calculated pKa: pKa1 3.6798572063446047 ± 0.5313097195517779; pKa2 5.490037117004395 ± 1.5246909375251723; 
Now calculating:
OC(CC1=CC=C(O)C=C1)C(O)=O


[query.py:297 - calculate_microstate_pka_values()] Using dimorphite-dl to identify protonation sites.


Proposed mol at pH 7.4: O=C([O-])C(O)Cc1ccc(O)cc1
Calculated pKa: pKa1 4.784922924041748 ± 0.39410791568190345; pKa2 8.262951545715332 ± 1.3586175080394756; pKa3 10.495713844299317 ± 0.60538507911415; 


In [ ]:
# @title Report SMILE list with pKa values

print("😀################################😀")
for i in range(len(protonation_states)):
    state = protonation_states[i]
    print(
        Chem.MolToSmiles(state.protonated_mol),
        Chem.MolToSmiles(state.deprotonated_mol),
    )
    print(state.pka)
print("😀################################😀")
